# Gaussian Naive Bayes: AQI Band Classifier
**Kathmandu air quality | Open-Meteo | Rashmi**

Matches the group brief:
- **Inputs (6):** temperature_2m, dew_point_2m, cloud_cover, precipitation, hour, month
- **Output:** AQI band, made from `us_aqi`
- **Split:** train Oct 2022 to Sep 2025, test Oct 2025 to Sep 2026 (by time, no shuffling)
- **Scaling:** StandardScaler, fitted on the training data only

Put `kathmandu_final.csv` in the same folder as this notebook (or upload it in Colab), then choose **Run all**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

## 1. Load data

In [ ]:
df = pd.read_csv("kathmandu_final.csv")
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)
print(df.shape, "| missing values:", int(df.isna().sum().sum()))

## 2. Time features

In [ ]:
df["hour"] = df["time"].dt.hour
df["month"] = df["time"].dt.month

## 3. AQI band from `us_aqi`
**Band cutoffs are not final.** The group agrees them at the meeting. Change only the `band()` function below.
Currently 3 bands, with the rare top bands merged (the 201+ band has only about 45 rows).

In [ ]:
def band(a):
    if a <= 100: return "Good/Moderate"
    if a <= 150: return "Unhealthy for Sensitive"
    return "Unhealthy"

df["band"] = df["us_aqi"].apply(band)
LABELS = ["Good/Moderate", "Unhealthy for Sensitive", "Unhealthy"]
df["band"].value_counts()

## 4. Inputs and time-based split

In [ ]:
FEATURES = ["temperature_2m", "dew_point_2m", "cloud_cover", "precipitation", "hour", "month"]

train = df[(df["time"] >= "2022-10-01") & (df["time"] < "2025-10-01")]
test  = df[df["time"] >= "2025-10-01"]

Xtr, ytr = train[FEATURES], train["band"]
Xte, yte = test[FEATURES],  test["band"]
print("Train rows:", len(Xtr), "|", train["time"].min(), "to", train["time"].max())
print("Test rows: ", len(Xte), "|", test["time"].min(), "to", test["time"].max())
print("\nBand share (%) in train vs test:")
print(pd.DataFrame({"train": ytr.value_counts(normalize=True) * 100,
                    "test": yte.value_counts(normalize=True) * 100}).round(1))

## 5. Sample rows (for the Data Table)

In [ ]:
df[["time", "temperature_2m", "dew_point_2m", "cloud_cover", "precipitation",
    "hour", "month", "us_aqi", "band"]].sample(5, random_state=1).sort_values("time")

## 6. Model: scaler + Gaussian Naive Bayes
The scaler sits inside a pipeline, so it is fitted on the training data only (also inside each validation fold).

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("nb", GaussianNB()),
])
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xte)

print("Default var_smoothing = 1e-9")
print("Accuracy:", round(accuracy_score(yte, pred), 4))
print("Macro F1:", round(f1_score(yte, pred, average="macro"), 4))
print("Always-guess-most-common baseline:", round(yte.value_counts(normalize=True).max(), 4))

## 7. Tuning `var_smoothing`
Validation is done **in time order inside the training years** (3 expanding folds). The test year is not touched.

In [ ]:
grid = {"nb__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]}
search = GridSearchCV(pipe, grid, cv=TimeSeriesSplit(n_splits=3), scoring="f1_macro")
search.fit(Xtr, ytr)

results = pd.DataFrame({
    "var_smoothing": grid["nb__var_smoothing"],
    "CV macro F1": search.cv_results_["mean_test_score"].round(4),
})
print(results.to_string(index=False))
print("\nBest var_smoothing:", search.best_params_["nb__var_smoothing"])
print("Best CV macro F1:  ", round(search.best_score_, 4))

## 8. Final test-year result (tuned model)

In [ ]:
best = search.best_estimator_          # already refit on all training data
pred_best = best.predict(Xte)

print("TEST YEAR (Oct 2025 to Sep 2026)")
print("Accuracy:", round(accuracy_score(yte, pred_best), 4))
print("Macro F1:", round(f1_score(yte, pred_best, average="macro"), 4))
print()
print(classification_report(yte, pred_best, labels=LABELS, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(yte, pred_best, labels=LABELS, cmap="Blues", xticks_rotation=20)
plt.title("Gaussian Naive Bayes: test year")
plt.tight_layout()
plt.show()

In [ ]:
# Predictions on the test year (first 8 rows)
out = test[["time", "us_aqi", "band"]].copy()
out["predicted"] = pred_best
out.head(8)

## Notes
- Naive Bayes assumes the inputs are independent and bell-shaped. This is only roughly true here (temperature and dew point are strongly related; precipitation is mostly zero).
- The test year is dirtier than the training years, which makes the task harder.
- If tuning gains are small, say so plainly in the report.